# Unified from-scratch stall experiment — seed 1

This notebook is one complete, self-contained replication. It needs only
CIFAR-100, Kaggle **T4 x2**, and the `github_token` secret. It does not load a
theta150/theta300 checkpoint and never rebases or restarts the learning-rate
schedule.

Phase 1 trains a randomly initialized CIFAR-ResNet18 with its metric-independent
base recipe: SGD for 200 recipe epochs, LR 0.1 with MultiStep drops at epochs
100 and 150 (`gamma=0.1`), momentum 0.9, and weight decay 5e-4. Every raw
validation best is fully checkpointed from the start. Once the base recipe is
complete, a stall is confirmed when exactly 100 epochs have elapsed since the
last raw validation best. Those 100 epochs are the matched Vanilla control and
Theta_P is that raw validation-best checkpoint.

Phase 2 forks that run's raw validation-best theta_P, including optimizer momentum,
scheduler position, RNG, loader stream, and data indices. GPU0 runs recurrent
E-driven O only. GPU1 runs scaled Bypass 70/30 and then launches
a fresh recurrent O-only process. Each method consumes 100 SGD epochs and writes per-epoch history,
latest/best checkpoints, diagnostics, timing, memory, and final/best accuracy.
Only `seed=1` differs from the other two generated notebooks.


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
RUN_ROOT = Path("/kaggle/working/unified_seed1_end_to_end_v4")
OUTPUT = RUN_ROOT / "phase2"

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                        url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, "main")
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONPATH=RUNTIME_PYTHONPATH)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO,
               env=test_env, check=True)
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
cifar_dirs = sorted({path.parent.resolve()
                     for path in Path("/kaggle/input").rglob("cifar-100-python")})
if not cifar_dirs: raise FileNotFoundError("Attach CIFAR-100")
DATA_ROOT = cifar_dirs[0]


In [ ]:
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from experiments.kaggle_checkpoint_discovery import discover_checkpoints
from experiments.plateau_protocol import BestCheckpointStallDetector

SEED = 1
PHASE1_OUTPUT = RUN_ROOT / "vanilla_stall"
PHASE1_OUTPUT.mkdir(parents=True, exist_ok=True)
schedule_id = "cifar-resnet18-sgd-multistep-200-v4-validation-best"

def copy_checkpoint(source, destination):
    source, destination = Path(source), Path(destination)
    if source.resolve() != destination.resolve():
        shutil.copy2(source, destination)

checkpoint_kinds = {
    "unified_vanilla_progress", "vanilla_validation_best_checkpoint",
    "vanilla_significant_best_checkpoint", "vanilla_exact_best_diagnostic",
    "plateau_fork_checkpoint"}
discovered, rejected_checkpoints = discover_checkpoints(
    "/kaggle/input", PHASE1_OUTPUT, kind=checkpoint_kinds)
progresses = [item for item in discovered
              if item["payload"].get("kind") == "unified_vanilla_progress"]
bests = [item for item in discovered
         if item["payload"].get("kind") ==
         "vanilla_validation_best_checkpoint"]
compatible = [item for item in progresses
              if item["payload"].get("protocol", {}).get("seed") == SEED
              and item["payload"].get("protocol", {}).get(
                  "schedule_id") == schedule_id]
PHASE1_MAX_EPOCH = 800
if not compatible:
    # A v3 run used the same training recipe but selected theta_P by trigger.
    # It is safe to migrate only when one attached full checkpoint lands
    # exactly on the raw validation-best epoch recorded in its history.
    legacy = [item for item in progresses
              if item["payload"].get("protocol", {}).get("seed") == SEED
              and item["payload"].get("protocol", {}).get(
                  "schedule_id") ==
                  "cifar-resnet18-sgd-multistep-200-v3-significant-fork"]
    if legacy:
        old_progress = max(
            legacy, key=lambda item: int(item["payload"]["epoch"]))
        old_payload = old_progress["payload"]
        old_history = old_payload["history"]
        old_best_row = max(
            old_history, key=lambda row: row["validation_accuracy"])
        old_best_epoch = int(old_best_row["epoch"])
        exact_snapshots = [item for item in discovered
                           if item["payload"].get("protocol", {}).get(
                               "seed") == SEED
                           and int(item["payload"].get("epoch", -1)) ==
                               old_best_epoch]
        if exact_snapshots:
            old_protocol = dict(old_payload["protocol"])
            old_protocol.pop("exact_best_min_gain", None)
            old_protocol.pop("significant_min_gain", None)
            old_protocol.update({
                "schedule_id": schedule_id,
                "selection_metric":
                    "validation accuracy (3,000-sample split)",
                "evaluation_role":
                    "model selection and reporting; official test unused",
                "stall_gate": (
                    "base recipe complete and 100 epochs after raw "
                    "validation best"),
                "theta_P_scope": "global raw validation best",
                "validation_best_min_gain": 0.0,
            })
            detector = BestCheckpointStallDetector(
                patience=100, min_gain=0.0, require_arm=False,
                exact_best_patience=True)
            for row in old_history:
                detector.update(
                    int(row["epoch"]), float(row["validation_accuracy"]))
            migrated_progress = {
                **old_payload, "protocol": old_protocol,
                "best_stall_detector": detector.state_dict()}
            migrated_best = {
                **exact_snapshots[0]["payload"],
                "kind": "vanilla_validation_best_checkpoint",
                "protocol": old_protocol}
            torch.save(
                migrated_progress, PHASE1_OUTPUT / "checkpoint_latest.pt")
            torch.save(migrated_best, PHASE1_OUTPUT / "checkpoint_best.pt")
            torch.save(
                migrated_best, PHASE1_OUTPUT / "checkpoint_exact_best.pt")
            compatible = [{
                "path": PHASE1_OUTPUT / "checkpoint_latest.pt",
                "payload": migrated_progress,
                "source": f"migrated:{old_progress['source']}"}]
            bests = [{
                "path": PHASE1_OUTPUT / "checkpoint_best.pt",
                "payload": migrated_best,
                "source": "migrated-validation-best"}]
            print("Migrated compatible v3 trajectory at exact validation best",
                  old_best_epoch)
        else:
            print("Cannot migrate attached v3 run: validation-best epoch",
                  old_best_epoch, "has no full checkpoint snapshot")
if compatible:
    progress = max(compatible, key=lambda item: int(item["payload"]["epoch"]))
    payload = progress["payload"]
    history = payload["history"]
    best_row = max(history, key=lambda row: row["validation_accuracy"])
    best_epoch = int(best_row["epoch"])
    matching_bests = [item for item in bests
                      if item["payload"].get("protocol") == payload["protocol"]
                      and int(item["payload"].get("epoch", -1)) == best_epoch]
    if not matching_bests:
        if int(payload["epoch"]) != best_epoch:
            raise RuntimeError(
                f"Found resumable epoch {payload['epoch']} but no matching "
                f"validation-best checkpoint at epoch {best_epoch}")
        selected_best = progress
        print("Repairing best checkpoint from atomic latest checkpoint")
    else:
        selected_best = matching_bests[0]
    copy_checkpoint(progress["path"], PHASE1_OUTPUT / "checkpoint_latest.pt")
    copy_checkpoint(
        selected_best["path"], PHASE1_OUTPUT / "checkpoint_best.pt")
    copy_checkpoint(
        selected_best["path"], PHASE1_OUTPUT / "checkpoint_exact_best.pt")
    remaining = max(0, best_epoch + 100 - int(payload["epoch"]))
    PHASE1_MAX_EPOCH = max(
        PHASE1_MAX_EPOCH, int(payload["epoch"]) + remaining)
    print("Resuming Phase 1:", {
        "checkpoint_epoch": int(payload["epoch"]),
        "validation_best_epoch": best_epoch,
        "minimum_additional_epochs_if_no_new_best": remaining,
        "source": progress["source"],
    })
else:
    print("No compatible Phase-1 checkpoint in /kaggle/input; starting epoch 0")


In [ ]:
SEED = 1
PHASE1_OUTPUT = RUN_ROOT / "vanilla_stall"
PHASE1_OUTPUT.mkdir(parents=True, exist_ok=True)

phase1_command = [
    sys.executable, "-m", "experiments.run_unified_vanilla_to_stall",
    "--reference-root", str(REFERENCE), "--data-root", str(DATA_ROOT),
    "--output", str(PHASE1_OUTPUT), "--seed", str(SEED),
    "--max-epoch", str(PHASE1_MAX_EPOCH), "--batch-size", "64",
    "--validation-samples", "5000", "--trigger-samples", "2000",
    "--tuning-samples", "128", "--lr", "0.1", "--recipe-epochs", "200",
    "--lr-milestones", "100,150", "--lr-gamma", "0.1",
    "--weight-decay", "0.0005", "--stall-patience", "100"]

phase1_log = (PHASE1_OUTPUT / "notebook_stream.log").open("a", buffering=1)
phase1_env = os.environ.copy()
phase1_env.update(CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1",
                  PYTHONPATH=RUNTIME_PYTHONPATH)
phase1_process = subprocess.Popen(
    phase1_command, cwd=REPO, env=phase1_env, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in phase1_process.stdout:
    phase1_log.write(line); phase1_log.flush()
    print(f"[seed{SEED} vanilla] {line}", end="", flush=True)
phase1_code = phase1_process.wait()
phase1_log.close()
if phase1_code:
    raise RuntimeError(f"unified Vanilla exited with code {phase1_code}")


In [ ]:
phase1_result = json.loads(
    (PHASE1_OUTPUT / "result.json").read_text())
if not phase1_result["plateau_found"]:
    archive = shutil.make_archive(
        str(RUN_ROOT), "gztar", root_dir=RUN_ROOT)
    raise RuntimeError(
        "No confirmed 100-epoch stall by max_epoch. All resumable Phase-1 "
        f"checkpoints were retained in {archive}")

PLATEAU_CHECKPOINT = Path(phase1_result["plateau_checkpoint"])
PLATEAU_PAYLOAD = torch.load(
    PLATEAU_CHECKPOINT, map_location="cpu", weights_only=False)
if PLATEAU_PAYLOAD.get("kind") != "plateau_fork_checkpoint":
    raise RuntimeError("Phase 1 did not produce a plateau fork checkpoint")
PLATEAU_HASH = hashlib.sha256(
    PLATEAU_CHECKPOINT.read_bytes()).hexdigest()
PLATEAU_EPOCH = int(PLATEAU_PAYLOAD["epoch"])
VANILLA_CONTROL = dict(PLATEAU_PAYLOAD["vanilla_control"])
if VANILLA_CONTROL["post_fork_epochs"] != 100:
    raise RuntimeError("Vanilla control is not exactly 100 observed epochs")
if VANILLA_CONTROL["role"] != "matched_validation_best_to_100_epoch_window":
    raise RuntimeError("Phase-1 Vanilla control has an invalid role")
if int(PLATEAU_PAYLOAD["protocol"]["seed"]) != SEED:
    raise RuntimeError("theta_best seed mismatch")
print("theta_best:", PLATEAU_EPOCH, PLATEAU_HASH)
print(json.dumps(VANILLA_CONTROL, indent=2, sort_keys=True))


## Phase 2 — three method forks plus matched Vanilla history

The Phase-1 window is exactly 100 epochs after raw validation-best theta_P and is
the Vanilla control. Three method jobs load byte-identical theta_P. GPU0 runs
E-driven O only; GPU1 runs Bypass then O-only.


In [ ]:
import threading

OUTPUT = RUN_ROOT / "phase2"
OUTPUT.mkdir(parents=True, exist_ok=True)


In [ ]:
def base_args(output):
    return [
        "--reference-root", str(REFERENCE), "--data-root", str(DATA_ROOT),
        "--plateau-checkpoint", str(PLATEAU_CHECKPOINT),
        "--plateau-checkpoint-hash", PLATEAU_HASH,
        "--output", str(output), "--post-fork-epochs", "100",
        "--seed", "1", "--batch-size", "64", "--rank", "4",
        "--opt1-epochs", "70", "--max-opt2-epochs", "30",
        "--probe-epsilon", "0.05", "--where-batches", "3",
        "--retrigger-patience", "10",
        "--significant-improvement", "0.001",
        "--gamma-increase-opt2-epoch", "15",
        "--gamma-post-increase-multiplier", "2.0",
        "--line-search-scales", "0.0125,0.025,0.05"]

commands = {
    name: [sys.executable, "-m", "experiments.run_plateau_fork",
           "--method", name] + base_args(OUTPUT / name)
    for name in ("bypass", "ours_e_driven_o", "o_projection_only")}

# A newly started Kaggle session can resume arm-level progress from an attached
# prior output. Only checkpoints with the same theta_P hash and full protocol
# are paired; unrelated seeds/runs are ignored.
arm_checkpoints, rejected_arm_checkpoints = discover_checkpoints(
    "/kaggle/input", OUTPUT,
    kind={"plateau_fork_arm_progress", "plateau_fork_arm_best"})
arm_progress = [item for item in arm_checkpoints
                if item["payload"]["kind"] == "plateau_fork_arm_progress"]
arm_bests = [item for item in arm_checkpoints
             if item["payload"]["kind"] == "plateau_fork_arm_best"]
for name in ("bypass", "ours_e_driven_o", "o_projection_only"):
    matches = [item for item in [*arm_progress, *arm_bests]
               if item["payload"].get("theta_best_hash") == PLATEAU_HASH
               and item["payload"].get("protocol", {}).get("method") == name]
    if not matches:
        continue
    progress = max(
        matches, key=lambda item: int(item["payload"]["post_fork_epoch"]))
    best_matches = [item for item in arm_bests
                    if item["payload"].get("theta_best_hash") == PLATEAU_HASH
                    and item["payload"].get("protocol") ==
                    progress["payload"].get("protocol")]
    if not best_matches:
        raise RuntimeError(
            f"Found resumable {name} progress without its best checkpoint")
    best = max(
        best_matches,
        key=lambda item: int(item["payload"]["post_fork_epoch"]))
    arm_output = OUTPUT / name
    arm_output.mkdir(parents=True, exist_ok=True)
    shutil.copy2(progress["path"], arm_output / "checkpoint_latest.pt")
    shutil.copy2(best["path"], arm_output / "checkpoint_best.pt")
    print(f"Resuming {name} at post-fork epoch "
          f"{progress['payload']['post_fork_epoch']}")

def stream(name, process, log):
    for line in process.stdout:
        log.write(line); log.flush()
        print(f"[{name}] {line}", end="", flush=True)

def launch(gpu, name):
    out = OUTPUT / name; out.mkdir(parents=True, exist_ok=True)
    log = (out / "run.log").open("a", buffering=1)
    env = os.environ.copy()
    env.update(CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1",
               PYTHONPATH=RUNTIME_PYTHONPATH)
    process = subprocess.Popen(
        commands[name], cwd=REPO, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1)
    thread = threading.Thread(target=stream, args=(name, process, log), daemon=True)
    thread.start()
    print(f"GPU{gpu}: started {name}, pid={process.pid}")
    return name, process, thread, log

def finish(job):
    name, process, thread, log = job
    code = process.wait()
    thread.join(); log.close()
    print(name, "exit=", code)
    return name, code

print("GPU0: E-driven O only")
print("GPU1: Bypass -> fresh O-only")
ours_job = launch(0, "ours_e_driven_o")
bypass_job = launch(1, "bypass")
bypass_status = finish(bypass_job)
if bypass_status[1]: raise RuntimeError(f"failed arm: {bypass_status}")
o_only_job = launch(1, "o_projection_only")
statuses = [finish(ours_job), finish(o_only_job)]
failures = [status for status in statuses if status[1]]
if failures: raise RuntimeError(f"failed arms: {failures}")


In [ ]:
results = {name: json.loads((OUTPUT / name / "result.json").read_text())
           for name in ("bypass", "ours_e_driven_o", "o_projection_only")}
for name, result in results.items():
    if result["plateau_checkpoint_hash"] != PLATEAU_HASH:
        raise RuntimeError(f"{name} used another theta_P")
    if result["theta_best_hash"] != PLATEAU_HASH:
        raise RuntimeError(f"{name} used another theta_best")
    if result["post_fork_epochs"] != 100:
        raise RuntimeError(f"{name} did not complete 100 epochs")
    if not Path(result["best_checkpoint"]).is_file():
        raise RuntimeError(f"{name} did not save checkpoint_best.pt")
VANILLA_CONTROL["plateau_checkpoint_hash"] = PLATEAU_HASH
VANILLA_CONTROL["theta_best_hash"] = PLATEAU_HASH
VANILLA_CONTROL["best_checkpoint"] = str(PLATEAU_CHECKPOINT)
results["vanilla"] = VANILLA_CONTROL
summary = {
    "plateau_epoch": PLATEAU_EPOCH,
    "plateau_checkpoint_hash": PLATEAU_HASH,
    "theta_best_hash": PLATEAU_HASH,
    "post_fork_epochs": 100, "official_test_used": False,
    "phase1_vanilla_control": VANILLA_CONTROL,
    "results": {name: {key: result.get(key) for key in (
        "fork_validation_accuracy", "fork_validation_loss",
        "theta_P_validation_accuracy", "theta_P_validation_loss",
        "meaningful_best_validation_accuracy",
        "fork_trigger_accuracy", "fork_trigger_loss",
        "exact_best_validation_accuracy",
        "significant_best_validation_accuracy",
        "final_validation_accuracy", "best_validation_accuracy",
        "final_validation_loss", "best_validation_loss",
        "validation_accuracy_delta", "best_validation_accuracy_delta",
        "epochs_to_best", "training_seconds", "peak_gpu_memory",
        "peak_train_params", "deploy_params", "time_spent_expanded_seconds",
        "bypass_completed", "contraction_at_projection",
        "opt1_epochs", "opt2_epochs", "train3_epochs",
        "gamma_increase_opt2_epoch", "gamma_post_increase_multiplier",
        "projection_loss_jump", "best_checkpoint", "intervention_count",
        "correction_application_count", "correction_application_rate",
        "intervention_seconds", "rollback_count", "retrigger_patience",
        "interventions")}
        for name, result in results.items()},
    "bypass_comparison_status": (
        "completed" if results["bypass"]["bypass_completed"] else
        "budget_exhausted_before_contraction_accuracy_is_diagnostic"),
}
(OUTPUT / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps(summary, indent=2, sort_keys=True))
print("\nmethod                  fork_acc   best_acc  final_acc  epoch_best")
for name in ("vanilla", "ours_e_driven_o", "bypass", "o_projection_only"):
    row = results[name]
    print(f"{name:23s} {row['fork_validation_accuracy']:.4f}     "
          f"{row['best_validation_accuracy']:.4f}    "
          f"{row['final_validation_accuracy']:.4f}    "
          f"{row['epochs_to_best']:>4}")
archive = shutil.make_archive(str(OUTPUT), "gztar", root_dir=OUTPUT)
print("Archive:", archive)


In [ ]:
full_archive = shutil.make_archive(
    str(RUN_ROOT), "gztar", root_dir=RUN_ROOT)
print("Complete seed archive:", full_archive)
print("Phase-1 latest:", PHASE1_OUTPUT / "checkpoint_latest.pt")
print("Phase-1 best:", PHASE1_OUTPUT / "checkpoint_best.pt")
print("Fork checkpoint:", PLATEAU_CHECKPOINT)
